<a href="https://colab.research.google.com/github/tusherkazi0-hub/-ucl-goal-scorer-analysis/blob/main/Ucl_goal_scorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("ucl_goal_scorers_raw.csv")

str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({"nan": np.nan, "None": np.nan, "": np.nan})

for c in ["Player", "Club", "Opponent", "Nationality"]:
    df[c] = df[c].str.title()

df["Round"] = df["Round"].str.title().replace({"R16": "Round Of 16"})
df["Goal_Type"] = df["Goal_Type"].str.title()
df["Venue"] = df["Venue"].str.title()

df["Minute"] = df["Minute"].astype(str).str.replace("'", "", regex=False)
df["Minute"] = pd.to_numeric(df["Minute"], errors="coerce")

df = df[df["Opponent"] != df["Club"]]
df = df.dropna(subset=["Opponent"])

median_minute = df["Minute"].median()
df["Minute"] = df["Minute"].fillna(median_minute)
df["Minute"] = df["Minute"].astype(int)

nat_map = df.dropna(subset=["Nationality"]).groupby("Player")["Nationality"].agg(lambda x: x.mode()[0])
df["Nationality"] = df.apply(
    lambda r: nat_map.get(r["Player"], r["Nationality"]) if pd.isna(r["Nationality"]) else r["Nationality"],
    axis=1
)

df = df.drop_duplicates()
df = df[(df["Minute"] >= 1) & (df["Minute"] <= 96)]

df["Goals_In_Match"] = pd.to_numeric(df["Goals_In_Match"], errors="coerce")
df = df.dropna(subset=["Goals_In_Match"])
df["Goals_In_Match"] = df["Goals_In_Match"].astype(int)

df = df.reset_index(drop=True)
df.to_csv("ucl_goal_scorers_clean.csv", index=False)
print(f"Clean rows: {len(df)}")


Clean rows: 1731


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

plt.rcParams["figure.dpi"] = 150
df = pd.read_csv("ucl_goal_scorers_clean.csv")

# ---------- 1. Feature Engineering: aggregate to one row per player ----------
knockout_rounds = ["Round Of 16", "Quarter-Final", "Semi-Final", "Final"]

def player_features(g):
    total_goals = len(g)
    gt = g["Goal_Type"].value_counts(normalize=True)
    return pd.Series({
        "Total_Goals": total_goals,
        "Pct_Penalty": gt.get("Penalty", 0) * 100,
        "Pct_Header": gt.get("Header", 0) * 100,
        "Pct_Right_Foot": gt.get("Right Foot", 0) * 100,
        "Pct_Left_Foot": gt.get("Left Foot", 0) * 100,
        "Pct_Free_Kick": gt.get("Free Kick", 0) * 100,
        "Pct_Home": (g["Venue"] == "Home").mean() * 100,
        "Avg_Minute": g["Minute"].mean(),
        "Pct_Knockout": g["Round"].isin(knockout_rounds).mean() * 100,
    })

player_df = df.groupby("Player").apply(player_features, include_groups=False).reset_index()

# Keep only players with a reasonable sample size for stable percentages
MIN_GOALS = 15
player_df = player_df[player_df["Total_Goals"] >= MIN_GOALS].reset_index(drop=True)
print(f"Players included in clustering (>= {MIN_GOALS} goals): {len(player_df)}")

feature_cols = ["Pct_Penalty", "Pct_Header", "Pct_Right_Foot", "Pct_Left_Foot",
                 "Pct_Free_Kick", "Pct_Home", "Avg_Minute", "Pct_Knockout"]
X = player_df[feature_cols].values

# ---------- 2. Standardization ----------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------- 3. Elbow Method + Silhouette Score to choose k ----------
inertias = []
sil_scores = []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertias, marker="o", color="#1f4e8c")
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.tight_layout()
plt.savefig("chart_elbow.png")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), sil_scores, marker="o", color="#27ae60")
plt.title("Silhouette Score by k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.savefig("chart_silhouette.png")
plt.close()

best_k = list(k_range)[int(np.argmax(sil_scores))]
print("Silhouette scores:", dict(zip(k_range, [round(s, 3) for s in sil_scores])))
print(f"Chosen k (best silhouette): {best_k}")

# ---------- 4. Final K-Means model ----------
K = 4 if 4 in k_range else best_k  # use k=4 as planned in proposal, matches typical elbow point
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
player_df["Cluster"] = kmeans.fit_predict(X_scaled)
final_sil = silhouette_score(X_scaled, player_df["Cluster"])
print(f"Final model k={K}, Silhouette Score={final_sil:.3f}")

# ---------- 5. PCA for 2D visualization ----------
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
player_df["PC1"] = coords[:, 0]
player_df["PC2"] = coords[:, 1]
explained_var = pca.explained_variance_ratio_
print(f"PCA explained variance: PC1={explained_var[0]:.2%}, PC2={explained_var[1]:.2%}")

# ---------- 6. Interpret & label clusters based on feature means ----------
cluster_profile = player_df.groupby("Cluster")[feature_cols + ["Total_Goals"]].mean().round(1)
print("\nCluster feature means:\n", cluster_profile)

def label_cluster(row):
    if row["Pct_Penalty"] == cluster_profile["Pct_Penalty"].max():
        return "Penalty Specialists"
    if row["Pct_Header"] == cluster_profile["Pct_Header"].max():
        return "Aerial / Header Threats"
    if row["Pct_Knockout"] == cluster_profile["Pct_Knockout"].max():
        return "Big-Game / Knockout Scorers"
    return "Balanced All-Rounders"

cluster_labels = {c: label_cluster(cluster_profile.loc[c]) for c in cluster_profile.index}
player_df["Cluster_Label"] = player_df["Cluster"].map(cluster_labels)
print("\nCluster labels:", cluster_labels)

# ---------- 7. Visualization: PCA scatter plot with cluster labels ----------
plt.figure(figsize=(9, 7))
colors = ["#1f4e8c", "#c0392b", "#27ae60", "#f39c12", "#8e44ad", "#16a085"]
for c in sorted(player_df["Cluster"].unique()):
    sub = player_df[player_df["Cluster"] == c]
    plt.scatter(sub["PC1"], sub["PC2"], s=90, color=colors[c % len(colors)],
                label=f"Cluster {c}: {cluster_labels[c]}", edgecolor="white", linewidth=0.6)
    for _, r in sub.iterrows():
        plt.annotate(r["Player"], (r["PC1"], r["PC2"]), fontsize=7, alpha=0.8,
                     xytext=(4, 4), textcoords="offset points")

plt.title("Player Scoring-Profile Clusters (PCA 2D Projection)")
plt.xlabel(f"PC1 ({explained_var[0]:.1%} variance)")
plt.ylabel(f"PC2 ({explained_var[1]:.1%} variance)")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig("chart_clusters_pca.png")
plt.close()

# ---------- 8. Save outputs ----------
player_df_sorted = player_df.sort_values(["Cluster", "Total_Goals"], ascending=[True, False])
player_df_sorted.to_csv("player_cluster_results.csv", index=False)

with open("clustering_summary.txt", "w") as f:
    f.write(f"Players included (>= {MIN_GOALS} goals): {len(player_df)}\n")
    f.write(f"Chosen k = {K}\n")
    f.write(f"Silhouette Score = {final_sil:.3f}\n")
    f.write(f"PCA explained variance: PC1={explained_var[0]:.2%}, PC2={explained_var[1]:.2%}\n\n")
    f.write("Cluster feature means:\n")
    f.write(cluster_profile.to_string())
    f.write("\n\nCluster labels:\n")
    for c, lbl in cluster_labels.items():
        members = player_df_sorted[player_df_sorted["Cluster"] == c]["Player"].tolist()
        f.write(f"Cluster {c} - {lbl}: {members}\n")

print("\nAll outputs saved: player_cluster_results.csv, clustering_summary.txt, charts.")


Players included in clustering (>= 15 goals): 40
Silhouette scores: {2: np.float64(0.141), 3: np.float64(0.147), 4: np.float64(0.141), 5: np.float64(0.14), 6: np.float64(0.171), 7: np.float64(0.161)}
Chosen k (best silhouette): 6
Final model k=4, Silhouette Score=0.141
PCA explained variance: PC1=23.27%, PC2=21.44%

Cluster feature means:
          Pct_Penalty  Pct_Header  Pct_Right_Foot  Pct_Left_Foot  \
Cluster                                                           
0               23.6        20.2            30.9           10.0   
1               25.2        34.1            21.2            9.1   
2               30.7        24.9            17.8           18.6   
3               28.5        22.4            23.3           14.8   

         Pct_Free_Kick  Pct_Home  Avg_Minute  Pct_Knockout  Total_Goals  
Cluster                                                                  
0                 15.3      44.6        49.3          61.0         43.2  
1                 10.3      47.8 

In [ ]:
import random
import csv

random.seed(42)

players = [
    ("Cristiano Ronaldo", "Portugal", ["Manchester United", "Real Madrid", "Juventus"]),
    ("Lionel Messi", "Argentina", ["Barcelona", "Paris Saint-Germain"]),
    ("Robert Lewandowski", "Poland", ["Borussia Dortmund", "Bayern Munich", "Barcelona"]),
    ("Karim Benzema", "France", ["Real Madrid"]),
    ("Raul Gonzalez", "Spain", ["Real Madrid"]),
    ("Ruud van Nistelrooy", "Netherlands", ["Manchester United", "Real Madrid"]),
    ("Thomas Muller", "Germany", ["Bayern Munich"]),
    ("Kylian Mbappe", "France", ["Paris Saint-Germain", "Real Madrid"]),
    ("Alfredo Di Stefano", "Argentina", ["Real Madrid"]),
    ("Neymar Jr", "Brazil", ["Barcelona", "Paris Saint-Germain"]),
    ("Zlatan Ibrahimovic", "Sweden", ["Inter Milan", "Barcelona", "AC Milan", "Paris Saint-Germain"]),
    ("Andriy Shevchenko", "Ukraine", ["AC Milan", "Chelsea"]),
    ("Mohamed Salah", "Egypt", ["Liverpool"]),
    ("Erling Haaland", "Norway", ["Borussia Dortmund", "Manchester City"]),
    ("Sergio Aguero", "Argentina", ["Manchester City"]),
    ("Filippo Inzaghi", "Italy", ["AC Milan"]),
    ("Didier Drogba", "Ivory Coast", ["Chelsea"]),
    ("Wayne Rooney", "England", ["Manchester United"]),
    ("Alessandro Del Piero", "Italy", ["Juventus"]),
    ("Luis Suarez", "Uruguay", ["Barcelona", "Atletico Madrid"]),
    ("Fernando Torres", "Spain", ["Liverpool", "Chelsea"]),
    ("Antoine Griezmann", "France", ["Atletico Madrid"]),
    ("Harry Kane", "England", ["Tottenham Hotspur", "Bayern Munich"]),
    ("Kevin De Bruyne", "Belgium", ["Manchester City"]),
    ("Sadio Mane", "Senegal", ["Liverpool", "Bayern Munich"]),
    ("Gareth Bale", "Wales", ["Real Madrid"]),
    ("Toni Kroos", "Germany", ["Real Madrid"]),
    ("Luka Modric", "Croatia", ["Real Madrid"]),
    ("Vinicius Junior", "Brazil", ["Real Madrid"]),
    ("Julian Alvarez", "Argentina", ["Manchester City"]),
    ("Jude Bellingham", "England", ["Real Madrid"]),
    ("Bukayo Saka", "England", ["Arsenal"]),
    ("Victor Osimhen", "Nigeria", ["Napoli"]),
    ("Ousmane Dembele", "France", ["Barcelona", "Paris Saint-Germain"]),
    ("Joshua Kimmich", "Germany", ["Bayern Munich"]),
    ("Bernardo Silva", "Portugal", ["Manchester City"]),
    ("Riyad Mahrez", "Algeria", ["Manchester City"]),
    ("Paulo Dybala", "Argentina", ["Juventus"]),
    ("Edinson Cavani", "Uruguay", ["Paris Saint-Germain"]),
    ("Memphis Depay", "Netherlands", ["Barcelona"]),
]

seasons = [f"{y}-{str(y+1)[2:]}" for y in range(2010, 2024)]
rounds = ["Group Stage", "Round of 16", "Quarter-final", "Semi-final", "Final", "Group stage", "GROUP STAGE", "R16"]
goal_types = ["Right Foot", "Left Foot", "Header", "Penalty", "Free Kick", "right foot", "PENALTY", "header"]
venues = ["Home", "Away", "home", "AWAY", "Home ", " Away"]

opponents = ["Bayern Munich", "Real Madrid", "Barcelona", "Manchester United", "Manchester City",
             "Liverpool", "Chelsea", "Juventus", "AC Milan", "Inter Milan", "Paris Saint-Germain",
             "Borussia Dortmund", "Atletico Madrid", "Arsenal", "Tottenham Hotspur", "Porto",
             "Ajax", "Napoli", "Sevilla", "Benfica", None, ""]

rows = []
for i in range(2000):
    name, nat, clubs = random.choice(players)
    club = random.choice(clubs)
    season = random.choice(seasons)
    rnd = random.choice(rounds)
    minute = random.randint(1, 96)
    gtype = random.choice(goal_types)
    venue = random.choice(venues)
    opp = random.choice(opponents)
    goals_in_match = random.choice([1, 1, 1, 1, 2, 2, 3])

    player_name = name
    if random.random() < 0.05:
        player_name = "  " + player_name.upper() + "  "
    if random.random() < 0.03:
        player_name = name.lower()

    minute_val = str(minute)
    if random.random() < 0.04:
        minute_val = f"{minute}'"
    if random.random() < 0.02:
        minute_val = ""

    club_val = club
    if random.random() < 0.03:
        club_val = club + "  "

    nat_val = nat
    if random.random() < 0.02:
        nat_val = None

    rows.append({
        "Player": player_name,
        "Nationality": nat_val,
        "Club": club_val,
        "Season": season,
        "Round": rnd,
        "Opponent": opp,
        "Minute": minute_val,
        "Goal_Type": gtype,
        "Venue": venue,
        "Goals_In_Match": goals_in_match,
    })

dupes = random.sample(rows, 40)
rows.extend(dupes)
random.shuffle(rows)

with open("ucl_goal_scorers_raw.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Total rows written: {len(rows)}")


Total rows written: 2040
